# SASRec Attention-Bias Time-Aware BPI2012 Colab Train (`refine_ml50_do035` baseline)

Colab notebook for the first attention-bias time-aware experiment on top of `refine_ml50_do035`.

Design:
- baseline reuse: `refine_ml50_do035`
- time source: `delta_start_seconds`
- pairwise causal gap attention bias
- 9-bucket scalar attention bias
- evaluate under both `NDCG@10` and `NDCG@5` model-selection criteria


In [14]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.10.0+cu128
cuda available: True
gpu name: NVIDIA L4


In [15]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
BASELINE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5'
TIMEAWARE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10'
TIMEAWARE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('BASELINE_NDCG5_OUTPUT_DIR:', BASELINE_NDCG5_OUTPUT_DIR)
print('TIMEAWARE_NDCG10_OUTPUT_DIR:', TIMEAWARE_NDCG10_OUTPUT_DIR)
print('TIMEAWARE_NDCG5_OUTPUT_DIR:', TIMEAWARE_NDCG5_OUTPUT_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
BASELINE_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5
TIMEAWARE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10
TIMEAWARE_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5


In [17]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$BASELINE_NDCG5_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG10_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG5_OUTPUT_DIR"


In [18]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 10 (delta 8), reused 10 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 8.28 KiB | 2.07 MiB/s, done.
From https://github.com/hwbuzz/time-aware-behavior-prediction
   4e5059d..0856655  main       -> origin/main
Updating 4e5059d..0856655
Fast-forward
 ...c_timeaware_bpi2012_colab_train_06_260514.ipynb |   1 +
 ...re_refine_ml50_do035_attention_bias_notebook.py | 415 +++++++++++++++++++++
 src/sasrec_model.py                                |  41 +-
 src/sasrec_utils.py                                |  42 ++-
 src/train_sasrec.py                                |  19 +
 5 files changed, 512 insertions(+), 6 deletions(-)
 create mode 100644 notebooks/sasrec_timeaware_bpi2012_colab_train_06_260514.ipynb
 create mode 100644 scripts/generate_timeaware_refine_ml50_do035_attention_bias

In [19]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [7]:
!pip install -r requirements_colab.txt


In [21]:
!ls "$DATA_DIR"


events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [20]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


## Experiment design

Fixed baseline setting:
- `refine_ml50_do035`
- `hidden_units=50, num_blocks=2, num_heads=1, maxlen=50, lr=0.001, dropout=0.35`
- seeds: `42`, `2024`, `7`

Time-aware design:
- `delta_start_seconds`
- causal pairwise gap attention bias
- 9-bucket scalar bias
- no additive time embedding in this notebook


## Check existing baseline runs


In [22]:
from pathlib import Path

baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]

for label, output_dir in [
    ('Baseline NDCG@10', Path(BASELINE_NDCG10_OUTPUT_DIR)),
    ('Baseline NDCG@5', Path(BASELINE_NDCG5_OUTPUT_DIR)),
]:
    print('=' * 80)
    print(label)
    for run_name in baseline_runs:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


Baseline NDCG@10
refine_ml50_do035_s42 EXISTS
refine_ml50_do035_s2024 EXISTS
refine_ml50_do035_s7 EXISTS
Baseline NDCG@5
refine_ml50_do035_s42 EXISTS
refine_ml50_do035_s2024 EXISTS
refine_ml50_do035_s7 EXISTS


## Check planned new runs


In [23]:
planned_ndcg10 = [
    'attnbias_dstart_ml50_do035_b9_s42',
    'attnbias_dstart_ml50_do035_b9_s2024',
    'attnbias_dstart_ml50_do035_b9_s7',
]
planned_ndcg5 = [
    'attnbias_dstart_ml50_do035_b9_s42',
    'attnbias_dstart_ml50_do035_b9_s2024',
    'attnbias_dstart_ml50_do035_b9_s7',
]

for label, output_dir, run_names in [
    ('Attention-Bias NDCG@10', Path(TIMEAWARE_NDCG10_OUTPUT_DIR), planned_ndcg10),
    ('Attention-Bias NDCG@5', Path(TIMEAWARE_NDCG5_OUTPUT_DIR), planned_ndcg5),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Attention-Bias NDCG@10
attnbias_dstart_ml50_do035_b9_s42 OK
attnbias_dstart_ml50_do035_b9_s2024 OK
attnbias_dstart_ml50_do035_b9_s7 OK
Attention-Bias NDCG@5
attnbias_dstart_ml50_do035_b9_s42 OK
attnbias_dstart_ml50_do035_b9_s2024 OK
attnbias_dstart_ml50_do035_b9_s7 OK


## Train attention-bias runs for `NDCG@10`


### attnbias_dstart_ml50_do035_b9_s42


In [24]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b9_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10/attnbias_dstart_ml50_do035_b9_s42
epoch=1, loss=0.5936
epoch=2, loss=0.2728
epoch=3, loss=0.2056
epoch=4, loss=0.1712
epoch=5, loss=0.1476
valid [full], NDCG@5: 0.6323, HR@5: 0.7273, NDCG@10: 0.6817, HR@10: 0.8820, MRR: 0.6319
valid [sampled], NDCG@5: 0.5579, HR@5: 0.5589, NDCG@10: 0.5653, HR@10: 0.5833, MRR: 0.5735
test [full], NDCG@5: 0.5731, HR@5: 0.7451, NDCG@10: 0.6023, HR@10: 0.8396, MRR: 0.5400
test [sampled], NDCG@5: 0.1993, HR@5: 0.2006, NDCG@10: 0.2175, HR@10: 0.2605, MRR: 0.2402
saved eval checkpoint: /content/drive/MyDrive/ai-projects/ti

### attnbias_dstart_ml50_do035_b9_s2024


In [25]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b9_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10/attnbias_dstart_ml50_do035_b9_s2024
epoch=1, loss=0.6139
epoch=2, loss=0.2744
epoch=3, loss=0.2006
epoch=4, loss=0.1654
epoch=5, loss=0.1461
valid [full], NDCG@5: 0.6635, HR@5: 0.7946, NDCG@10: 0.7241, HR@10: 0.9791, MRR: 0.6490
valid [sampled], NDCG@5: 0.5612, HR@5: 0.5656, NDCG@10: 0.5697, HR@10: 0.5924, MRR: 0.5772
test [full], NDCG@5: 0.7707, HR@5: 0.9296, NDCG@10: 0.7940, HR@10: 1.0000, MRR: 0.7285
test [sampled], NDCG@5: 0.2083, HR@5: 0.2682, NDCG@10: 0.2660, HR@10: 0.4467, MRR: 0.2421
saved eval checkpoint: /content/drive/MyDrive/ai-projects/

### attnbias_dstart_ml50_do035_b9_s7


In [26]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b9_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10/attnbias_dstart_ml50_do035_b9_s7
epoch=1, loss=0.5867
epoch=2, loss=0.2771
epoch=3, loss=0.2056
epoch=4, loss=0.1732
epoch=5, loss=0.1501
valid [full], NDCG@5: 0.6290, HR@5: 0.7278, NDCG@10: 0.6913, HR@10: 0.9195, MRR: 0.6303
valid [sampled], NDCG@5: 0.5544, HR@5: 0.5594, NDCG@10: 0.5641, HR@10: 0.5900, MRR: 0.5691
test [full], NDCG@5: 0.6675, HR@5: 0.8424, NDCG@10: 0.6993, HR@10: 0.9423, MRR: 0.6256
test [sampled], NDCG@5: 0.2334, HR@5: 0.2392, NDCG@10: 0.2791, HR@10: 0.3871, MRR: 0.2815
saved eval checkpoint: /content/drive/MyDrive/ai-projects/tim

## Train attention-bias runs for `NDCG@5`


### attnbias_dstart_ml50_do035_b9_s42


In [27]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b9_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5/attnbias_dstart_ml50_do035_b9_s42
epoch=1, loss=0.5936
epoch=2, loss=0.2727
epoch=3, loss=0.2056
epoch=4, loss=0.1713
epoch=5, loss=0.1476
valid [full], NDCG@5: 0.6324, HR@5: 0.7276, NDCG@10: 0.6819, HR@10: 0.8820, MRR: 0.6321
valid [sampled], NDCG@5: 0.5579, HR@5: 0.5589, NDCG@10: 0.5654, HR@10: 0.5834, MRR: 0.5736
test [full], NDCG@5: 0.5722, HR@5: 0.7453, NDCG@10: 0.6007, HR@10: 0.8397, MRR: 0.5381
test [sampled], NDCG@5: 0.1982, HR@5: 0.1995, NDCG@10: 0.2166, HR@10: 0.2600, MRR: 0.2390
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time

### attnbias_dstart_ml50_do035_b9_s2024


In [28]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b9_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5/attnbias_dstart_ml50_do035_b9_s2024
epoch=1, loss=0.6139
epoch=2, loss=0.2744
epoch=3, loss=0.2006
epoch=4, loss=0.1654
epoch=5, loss=0.1460
valid [full], NDCG@5: 0.6628, HR@5: 0.7938, NDCG@10: 0.7237, HR@10: 0.9785, MRR: 0.6487
valid [sampled], NDCG@5: 0.5611, HR@5: 0.5654, NDCG@10: 0.5695, HR@10: 0.5920, MRR: 0.5771
test [full], NDCG@5: 0.7702, HR@5: 0.9285, NDCG@10: 0.7942, HR@10: 1.0000, MRR: 0.7287
test [sampled], NDCG@5: 0.2082, HR@5: 0.2682, NDCG@10: 0.2660, HR@10: 0.4470, MRR: 0.2421
saved eval checkpoint: /content/drive/MyDrive/ai-projects/ti

### attnbias_dstart_ml50_do035_b9_s7


In [29]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b9_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5/attnbias_dstart_ml50_do035_b9_s7
epoch=1, loss=0.5867
epoch=2, loss=0.2771
epoch=3, loss=0.2056
epoch=4, loss=0.1732
epoch=5, loss=0.1501
valid [full], NDCG@5: 0.6290, HR@5: 0.7278, NDCG@10: 0.6913, HR@10: 0.9195, MRR: 0.6303
valid [sampled], NDCG@5: 0.5544, HR@5: 0.5594, NDCG@10: 0.5641, HR@10: 0.5900, MRR: 0.5691
test [full], NDCG@5: 0.6675, HR@5: 0.8424, NDCG@10: 0.6993, HR@10: 0.9423, MRR: 0.6256
test [sampled], NDCG@5: 0.2334, HR@5: 0.2392, NDCG@10: 0.2791, HR@10: 0.3871, MRR: 0.2815
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-

## Rebuild result tables


In [30]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'use_time_embedding': config.get('use_time_embedding', False),
            'use_time_attention_bias': config.get('use_time_attention_bias', False),
            'time_modeling_mode': config.get('time_modeling_mode'),
            'time_encoding': config.get('time_encoding'),
            'time_delta_column': config.get('time_delta_column'),
            'time_bucket_boundaries_parsed': config.get('time_bucket_boundaries_parsed'),
            'time_attention_bias_bucket_count': config.get('time_attention_bias_bucket_count'),
            'primary_metric_name': config.get('selection_metric'),
        }
        best_valid = summary.get('best_valid', {})
        best_test = summary.get('best_test_at_best_valid', {})
        def pick(metrics_group, mode, key):
            return metrics_group.get(mode, {}).get(key)
        row.update({
            'best_valid_full_ndcg@10': pick(best_valid, 'full', 'ndcg@10'),
            'best_valid_full_hr@10': pick(best_valid, 'full', 'hr@10'),
            'best_valid_full_ndcg@5': pick(best_valid, 'full', 'ndcg@5'),
            'best_valid_full_hr@5': pick(best_valid, 'full', 'hr@5'),
            'best_valid_full_mrr': pick(best_valid, 'full', 'mrr'),
            'best_test_full_ndcg@10': pick(best_test, 'full', 'ndcg@10'),
            'best_test_full_hr@10': pick(best_test, 'full', 'hr@10'),
            'best_test_full_ndcg@5': pick(best_test, 'full', 'ndcg@5'),
            'best_test_full_hr@5': pick(best_test, 'full', 'hr@5'),
            'best_test_full_mrr': pick(best_test, 'full', 'mrr'),
            'best_valid_sampled_ndcg@10': pick(best_valid, 'sampled', 'ndcg@10'),
            'best_valid_sampled_hr@10': pick(best_valid, 'sampled', 'hr@10'),
            'best_valid_sampled_ndcg@5': pick(best_valid, 'sampled', 'ndcg@5'),
            'best_valid_sampled_hr@5': pick(best_valid, 'sampled', 'hr@5'),
            'best_valid_sampled_mrr': pick(best_valid, 'sampled', 'mrr'),
            'best_test_sampled_ndcg@10': pick(best_test, 'sampled', 'ndcg@10'),
            'best_test_sampled_hr@10': pick(best_test, 'sampled', 'hr@10'),
            'best_test_sampled_ndcg@5': pick(best_test, 'sampled', 'ndcg@5'),
            'best_test_sampled_hr@5': pick(best_test, 'sampled', 'hr@5'),
            'best_test_sampled_mrr': pick(best_test, 'sampled', 'mrr'),
        })
        rows.append(row)
    return pd.DataFrame(rows)


In [31]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", None)

## NDCG@10 comparison summary


In [32]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
timeaware_runs = [
    'attnbias_dstart_ml50_do035_b9_s42',
    'attnbias_dstart_ml50_do035_b9_s2024',
    'attnbias_dstart_ml50_do035_b9_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG10_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['time_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['time_variant'] = 'attention_bias_dstart_b9'

df_ndcg10 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg10 = df_ndcg10.sort_values(['time_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg10[[
    'run_name', 'seed', 'time_variant', 'use_time_embedding', 'use_time_attention_bias', 'time_modeling_mode',
    'time_delta_column', 'time_bucket_boundaries_parsed', 'time_attention_bias_bucket_count',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,time_variant,use_time_embedding,use_time_attention_bias,time_modeling_mode,time_delta_column,time_bucket_boundaries_parsed,time_attention_bias_bucket_count,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,attnbias_dstart_ml50_do035_b9_s7,7,attention_bias_dstart_b9,False,True,attention_bias,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.755366,0.953861,0.714870,0.824671,0.699184,0.846762,1.000000,0.846665,0.999728,0.793978,0.623265,0.666077,0.607399,0.615479,0.623619,0.497216,0.628223,0.442098,0.450204,0.485290
1,attnbias_dstart_ml50_do035_b9_s42,42,attention_bias_dstart_b9,False,True,attention_bias,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.741856,0.969750,0.709565,0.864487,0.673783,0.785724,0.999865,0.764719,0.935724,0.714860,0.585005,0.635957,0.566501,0.577970,0.585189,0.199334,0.453522,0.115783,0.193920,0.156593
2,attnbias_dstart_ml50_do035_b9_s2024,2024,attention_bias_dstart_b9,False,True,attention_bias,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.735385,0.971475,0.702964,0.865254,0.665031,0.885738,0.999865,0.863911,0.928234,0.850981,0.566216,0.612548,0.548977,0.558247,0.568592,0.428086,0.626287,0.366083,0.436179,0.388188
3,refine_ml50_do035_s7,7,baseline,False,False,None,delta_prev_seconds,None,None,0.728826,0.961737,0.696223,0.857259,0.660451,0.872416,1.000000,0.859037,0.957678,0.831208,0.577254,0.611043,0.565802,0.575315,0.581977,0.403018,0.656933,0.324827,0.417493,0.345750
4,refine_ml50_do035_s42,42,baseline,False,False,None,delta_prev_seconds,None,None,0.735378,0.977276,0.697410,0.854186,0.663510,0.891774,1.000000,0.868726,0.926375,0.858375,0.568693,0.606441,0.555385,0.565093,0.572753,0.456024,0.640631,0.399455,0.467411,0.419230
5,refine_ml50_do035_s2024,2024,baseline,False,False,None,delta_prev_seconds,None,None,0.741573,0.992973,0.713154,0.906351,0.664353,0.775861,1.000000,0.727453,0.850196,0.707187,0.572972,0.597043,0.563603,0.567069,0.582973,0.257972,0.431769,0.203548,0.264514,0.234183


In [33]:
summary_ndcg10 = df_ndcg10.groupby('time_variant')[[
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg10


best_valid_full_ndcg@10           best_test_full_ndcg@10           best_valid_full_ndcg@5           best_test_full_ndcg@5           best_valid_full_mrr           best_test_full_mrr           best_valid_sampled_ndcg@10           best_test_sampled_ndcg@10           best_valid_sampled_ndcg@5           best_test_sampled_ndcg@5           best_valid_sampled_mrr           best_test_sampled_mrr          
                                            mean       std                   mean       std                   mean       std                  mean       std                mean       std               mean       std                       mean       std                      mean       std                      mean       std                     mean       std                   mean       std                  mean       std
time_variant                                                                                                                                                                                                                                                                                                                                                                                                                            
attention_bias_dstart_b9                0.744203  0.010195               0.839408  0.050411               0.709133  0.005965              0.825099  0.052996            0.679333  0.017740           0.786607  0.068359                   0.591495  0.029073                  0.374879  0.155906                  0.574292  0.029980                 0.307988  0.170739               0.592467  0.028226              0.343357  0.168872
baseline                                0.735259  0.006374               0.846684  0.062093               0.702263  0.009451              0.818405  0.078916            0.662771  0.002053           0.798923  0.080599                   0.572973  0.004280                  0.372338  0.102528                  0.561597  0.005491                 0.309277  0.098875               0.579235  0.005635              0.333054  0.093175

Interpretation guide for NDCG@10:
- compare `best_valid_full_ndcg@10` and `best_test_full_ndcg@10` first
- then check whether sampled and MRR move in the same direction
- baseline is reused; only attention-bias runs are newly trained here


## NDCG@5 comparison summary


In [34]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
timeaware_runs = [
    'attnbias_dstart_ml50_do035_b9_s42',
    'attnbias_dstart_ml50_do035_b9_s2024',
    'attnbias_dstart_ml50_do035_b9_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG5_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG5_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['time_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['time_variant'] = 'attention_bias_dstart_b9'

df_ndcg5 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg5 = df_ndcg5.sort_values(['time_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg5[[
    'run_name', 'seed', 'time_variant', 'use_time_embedding', 'use_time_attention_bias', 'time_modeling_mode',
    'time_delta_column', 'time_bucket_boundaries_parsed', 'time_attention_bias_bucket_count',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,time_variant,use_time_embedding,use_time_attention_bias,time_modeling_mode,time_delta_column,time_bucket_boundaries_parsed,time_attention_bias_bucket_count,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,attnbias_dstart_ml50_do035_b9_s7,7,attention_bias_dstart_b9,False,True,attention_bias,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.765708,0.975845,0.728865,0.854661,0.703877,0.847794,1.000000,0.847697,0.999728,0.795264,0.625614,0.676007,0.606813,0.616159,0.624051,0.500175,0.632429,0.444546,0.452782,0.487732
1,attnbias_dstart_ml50_do035_b9_s42,42,attention_bias_dstart_b9,False,True,attention_bias,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.741824,0.969750,0.709060,0.862317,0.673822,0.783187,0.999865,0.763183,0.935995,0.711912,0.585252,0.636770,0.566501,0.577970,0.585234,0.194366,0.445244,0.112266,0.190256,0.152804
2,attnbias_dstart_ml50_do035_b9_s2024,2024,attention_bias_dstart_b9,False,True,attention_bias,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.741407,0.975616,0.712340,0.881719,0.670798,0.894503,0.999865,0.878950,0.952884,0.861072,0.572542,0.619131,0.554208,0.560547,0.576476,0.456145,0.618822,0.406179,0.464726,0.428497
3,refine_ml50_do035_s7,7,baseline,False,False,None,delta_prev_seconds,None,None,0.728826,0.961737,0.696223,0.857259,0.660451,0.872416,1.000000,0.859037,0.957678,0.831208,0.577254,0.611043,0.565802,0.575315,0.581977,0.403018,0.656933,0.324827,0.417493,0.345750
4,refine_ml50_do035_s42,42,baseline,False,False,None,delta_prev_seconds,None,None,0.735378,0.977276,0.697410,0.854186,0.663510,0.891774,1.000000,0.868726,0.926375,0.858375,0.568693,0.606441,0.555385,0.565093,0.572753,0.456024,0.640631,0.399455,0.467411,0.419230
5,refine_ml50_do035_s2024,2024,baseline,False,False,None,delta_prev_seconds,None,None,0.741573,0.992973,0.713154,0.906351,0.664353,0.775861,1.000000,0.727453,0.850196,0.707187,0.572972,0.597043,0.563603,0.567069,0.582973,0.257972,0.431769,0.203548,0.264514,0.234183


In [35]:
summary_ndcg5 = df_ndcg5.groupby('time_variant')[[
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg5


best_valid_full_ndcg@10           best_test_full_ndcg@10           best_valid_full_ndcg@5           best_test_full_ndcg@5           best_valid_full_mrr           best_test_full_mrr           best_valid_sampled_ndcg@10           best_test_sampled_ndcg@10           best_valid_sampled_ndcg@5           best_test_sampled_ndcg@5           best_valid_sampled_mrr           best_test_sampled_mrr          
                                            mean       std                   mean       std                   mean       std                  mean       std                mean       std               mean       std                       mean       std                      mean       std                      mean       std                     mean       std                   mean       std                  mean       std
time_variant                                                                                                                                                                                                                                                                                                                                                                                                                            
attention_bias_dstart_b9                0.749646  0.013911               0.841828  0.055897               0.716755  0.010615              0.829944  0.059891            0.682833  0.018287           0.789416  0.074752                   0.594469  0.027711                  0.383562  0.165321                  0.575841  0.027518                 0.320997  0.181781               0.595254  0.025321              0.356344  0.178742
baseline                                0.735259  0.006374               0.846684  0.062093               0.702263  0.009451              0.818405  0.078916            0.662771  0.002053           0.798923  0.080599                   0.572973  0.004280                  0.372338  0.102528                  0.561597  0.005491                 0.309277  0.098875               0.579235  0.005635              0.333054  0.093175

Interpretation guide for NDCG@5:
- compare `best_valid_full_ndcg@5` and `best_test_full_ndcg@5` first
- then check whether sampled and MRR move in the same direction
- baseline is reused; only attention-bias runs are newly trained here
